In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import torch

#setting device

if torch.backends.mps.is_available():
    device = torch.device("mps")  # Apple GPU
    print("Using Apple MPS GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
matrix_likelihoods = torch.from_numpy(np.load('log_prob.npy')).to(device).T

In [ ]:
# =========================
# Funzione obiettivo
# =========================
def objective(y, r):
    """
    y: torch.Tensor shape (n,) (parametri liberi)
    r: torch.Tensor shape (n, m) (log-likelihood)
    """
    x = torch.softmax(y, dim=0)  # vettore nel simplex
    log_x = torch.log(x + 1e-16)  # stabilità
    log_terms = torch.logsumexp(log_x[:, None] + r, dim=0)  # log(sum_i x_i * exp(r_ij))
    return -torch.sum(log_terms)

In [ ]:
def ottimizza(r, sensibilita=1e-10, pazienza=100, max_iter=50000, verbose=True):
    """
    Esegue l'ottimizzazione con early stopping.
    
    Parametri:
        r: tensore di input
        sensibilita: miglioramento minimo richiesto nella loss per continuare
        pazienza: numero di iterazioni consecutive senza miglioramento dopo cui fermarsi
        max_iter: numero massimo di iterazioni
    """
    r = r.to(device=device)
    n = r.shape[0]
    y = torch.zeros(n, device=device, requires_grad=True)
    
    optimizer = torch.optim.Adam([y], lr=0.05)
    history = []
    
    best_loss = float('inf')
    counter = 0  # conta quante iterazioni senza miglioramento
    
    for it in range(max_iter):
        optimizer.zero_grad()
        loss = objective(y, r)
        loss.backward()
        optimizer.step()
        
        current_loss = loss.item()
        history.append(current_loss)
        
        # Controllo miglioramento
        if best_loss - current_loss > sensibilita:
            best_loss = current_loss
            counter = 0  # reset contatore
        else:
            counter += 1
        
        # Stampa di debug
        if ((it % 50 == 0) and (verbose)):
            print(f"Iter {it:4d} | Loss {current_loss:.6f} | Best {best_loss:.6f} | No improv: {counter}")
        
        # Early stopping
        if counter >= pazienza:
            if verbose:
                print(f"Early stopping a iterazione {it} — nessun miglioramento in {pazienza} step.")
            break
    
    return y, history

In [ ]:
plt.figure(figsize=(12, 6))
plt.imshow(matrix_likelihoods.cpu(), aspect='auto', cmap='viridis')
plt.colorbar(label='Log Probability')
plt.xlabel('Images')
plt.ylabel('Conformation index')
plt.title('Log probabilities for each theta × image')
plt.xlim(0,1000)
plt.show()

In [ ]:
ys, _ = ottimizza(matrix_likelihoods,verbose=True)
x_opt = torch.softmax(ys, dim=0).detach().cpu().numpy()
print("Optimal x:", x_opt)
print("Sum of x:", np.sum(x_opt))

In [ ]:
# Plot final weights
plt.plot(x_opt, 'o-')

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(x_opt, bins=100, color='skyblue', edgecolor='black')
plt.show()